# 08 · Dead letters, TTL, and bounded delayed retries

**Level:** intermediate · **Time:** 40 minutes · **Prerequisites:** 00, 02, and 07.

Build a work queue, a 500 ms retry queue, and a parking queue. A flaky job succeeds on its third attempt; a poison job is parked after three attempts.

```text
work --confirmed republish + ack--> retry (TTL 500 ms)
  ^                                  |
  +-------- dead-letter on expiry ----+
  |
  +-- reject(requeue=False) ----------> parking
```

A DLX is an ordinary exchange used when a message is dead-lettered. This lesson uses the default exchange (`""`) with queue-name routing keys. TTL is an expiry mechanism, not a precise scheduling guarantee. Do not create a rapid `nack(requeue=True)` loop.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course" / "rabbit.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.rabbit import connect, unique_name, wait_for_message, broker_url
import pika
print("Course root:", ROOT)


## Step 1 — Declare the three queues

Each queue is durable. The retry queue has no consumer: its messages wait until expiry and are routed back to work. The work queue dead-letters final failures to parking. Here arguments keep the example self-contained; operational policies are easier to update in deployed systems.


In [ ]:
import json
import time

work_queue = unique_name("retry.work")
retry_queue = unique_name("retry.delay")
parking_queue = unique_name("retry.parking")
with connect() as connection:
    channel = connection.channel()
    channel.queue_declare(queue=parking_queue, durable=True)
    channel.queue_declare(queue=work_queue, durable=True, arguments={
        "x-dead-letter-exchange": "", "x-dead-letter-routing-key": parking_queue,
    })
    channel.queue_declare(queue=retry_queue, durable=True, arguments={
        "x-message-ttl": 500,
        "x-dead-letter-exchange": "", "x-dead-letter-routing-key": work_queue,
    })
print("Topology ready:", work_queue, retry_queue, parking_queue, sep="\n")


## Step 2 — Publish two jobs with stable IDs

An application-owned `attempt` header is explicit and independent of broker redelivery flags. Retrying must preserve the job's identity. `x-death` will later show broker-generated history for the current publication, not an application-wide retry counter.


In [ ]:
with connect() as connection:
    channel = connection.channel()
    channel.confirm_delivery()
    for job_id in ("flaky", "poison"):
        channel.basic_publish(
            exchange="", routing_key=work_queue,
            body=json.dumps({"job_id": job_id}).encode(), mandatory=True,
            properties=pika.BasicProperties(
                delivery_mode=2, content_type="application/json",
                message_id=job_id, headers={"attempt": 1},
            ),
        )
print("Published flaky and poison")


## Step 3 — Process with a maximum of three attempts

Predict six deliveries: `flaky` three times and `poison` three times. A retry is confirmed **before** acknowledging the original. If publishing raises, the connection closes with the original unacknowledged. Each retry is a new application publication: copy application-owned headers only, and let RabbitMQ maintain its own death headers.

This is still not atomic: a crash between republishing and acknowledging can create duplicates. Use idempotency (09). Also, ordinary/classic-queue dead-lettering can lose a message if its destination is unavailable. For stronger transfer guarantees, configure a source quorum queue with `dead-letter-strategy=at-least-once`, `overflow=reject-publish`, and a DLX; capacity limits and duplicate handling still matter.


In [ ]:
MAX_ATTEMPTS = 3
attempts = []
completed = []
parked = []
deadline = time.monotonic() + 15
with connect() as connection:
    channel = connection.channel()
    channel.confirm_delivery()
    while len(completed) + len(parked) < 2:
        if time.monotonic() >= deadline:
            raise TimeoutError("Retry workflow did not finish; inspect queue bindings and TTL")
        try:
            method, properties, body = wait_for_message(channel, work_queue, timeout=1)
        except TimeoutError:
            continue
        job = json.loads(body)
        attempt = int(properties.headers["attempt"])
        attempts.append((job["job_id"], attempt))
        succeeds = job["job_id"] == "flaky" and attempt >= 3
        if succeeds:
            completed.append(job["job_id"])
            channel.basic_ack(method.delivery_tag)
            print(job["job_id"], attempt, "SUCCESS")
        elif attempt < MAX_ATTEMPTS:
            # Preserve application state, not broker-owned x-death / x-first-death-* / x-last-death-*.
            headers = {"attempt": attempt + 1}
            channel.basic_publish(
                exchange="", routing_key=retry_queue, body=body, mandatory=True,
                properties=pika.BasicProperties(
                    delivery_mode=2, content_type="application/json",
                    message_id=properties.message_id, headers=headers,
                ),
            )
            channel.basic_ack(method.delivery_tag)
            print(job["job_id"], attempt, "RETRY LATER")
        else:
            channel.basic_reject(method.delivery_tag, requeue=False)
            parked.append(job["job_id"])
            print(job["job_id"], attempt, "PARKED")
assert completed == ["flaky"] and parked == ["poison"]
assert sorted(attempts) == sorted((job, attempt) for job in ("flaky", "poison") for attempt in (1, 2, 3))


## Step 4 — Inspect the poison message and its history

Expected: `attempt=3`, a `rejected` death for the work queue, and an `expired` death for the retry queue. Both broker events happen after the final application republish: the retry queue expires that publication, then the work consumer rejects it into parking.

RabbitMQ compresses repeated broker-generated deaths by queue and reason within a publication's broker history. **RabbitMQ 4 does not interpret client-supplied `x-death` as trusted prior history when an AMQP 0.9.1 application republishes.** Do not expect its count to accumulate across all three application attempts. Our republisher deliberately excludes broker-owned death headers; the application-owned `attempt` header enforces the retry budget. See the [RabbitMQ maintainers' explanation](https://github.com/rabbitmq/rabbitmq-server/discussions/15386).

Parking preserves evidence for repair and a controlled replay; endlessly replaying poison messages only repeats the failure.


In [ ]:
with connect() as connection:
    channel = connection.channel()
    method, properties, body = wait_for_message(channel, parking_queue)
    history = properties.headers.get("x-death", [])
    assert json.loads(body)["job_id"] == "poison"
    assert properties.headers["attempt"] == 3
    assert any(entry["reason"] == "rejected" for entry in history)
    assert any(entry["reason"] == "expired" for entry in history)
    print("Parked body:", body.decode())
    print("Dead-letter history:", history)
    channel.basic_ack(method.delivery_tag)


## Exercise — Add backoff and classify failures

Design separate delay queues for 1 s, 5 s, and 30 s retries. Which failures should skip retries? What happens if the parking queue cannot accept a message?

**Suggested solution:** choose a delay queue by attempt number; keep one TTL per delay queue to make behavior easier to predict. Reject invalid schemas and impossible business operations directly to parking. Use a finite retry budget, idempotency, monitored parking queues, and an appropriate quorum dead-letter strategy where loss is unacceptable. Different per-message TTLs can wait behind other messages before they are dead-lettered; avoid treating a shared queue as an exact timer.

**Troubleshooting:** inspect the default-exchange routing key, which must exactly equal the destination queue name. A queue argument mismatch requires a new name or coordinated replacement. `x-death` absence before the first dead-letter event is normal.


## Cleanup


In [ ]:
with connect() as connection:
    channel = connection.channel()
    for name in (work_queue, retry_queue, parking_queue):
        channel.queue_delete(queue=name)
print("Deleted this lesson's three queues")


## References

- [Dead-letter exchanges and safety](https://www.rabbitmq.com/docs/4.2/dlx)
- [TTL behavior](https://www.rabbitmq.com/docs/4.2/ttl)
- [Quorum dead-letter guarantees](https://www.rabbitmq.com/docs/4.2/quorum-queues#dead-lettering)
